# Dia 02 - Armazenamento e Arquiteturas 
- Objetivo: Compreender o impacto dos formatos de armazenamento e como as arquiteturas evoluíram do DW e LH.

## SESSÕES
- Comparativo entre diferentes formatos de dados (.csv vs .parquet)
https://www.alura.com.br/artigos/arquivos-parquet
    - Apresentar uma comparação entre o tamanho dos arquivos em disco
    - Apresentar a comparação entre o tempo de leitura entre CSV e parquet

- Simulação de um Data Lake em pastas (medallion architecture / zones)
    - Demonstrar a organização dos arquivos em diretórios /raw (dados brutos) /silver (dados limpos em parquet) e /gold (dados agregados para negócio)

- Armazenamento colunar com DuckDB
    - Apresentação do conceito de armazenamento colunar
    - Apresentar o conceito de um banco OLAP colunar-in-process
    - bancos olap vs oltp

- Staging e Padrão ELT
    - Explicar brevemente as diferenças entre etl e elt
    - Demonstrar como criar schemas e realizar carga de dados brutos direto para tabelas de staging utilizando o duckDB
    - Exibir como os schemas ingeridos ficam dispostos em relação a estrutura e disponibilidade dos dados, bem como discutir potenciais inconsistências


## 1. Comparativo entre diferentes formatos de dados: CSV e Parquet

Para ilustrar a diferença teórica entre um formato orientado a linhas (CSV) e um formato colunar com compressão nativa (Parquet), vamos colocar ambos à prova utilizando um grande volume de dados.

O código a seguir criará, em tempo de execução, um dataset fictício contendo 1 milhão de registros de vendas. Com esse volume gerado, faremos duas medições:

- Espaço em Disco: Vamos salvar o mesmo conjunto de dados nos formatos .csv e .parquet e comparar o tamanho (em MB) dos arquivos gerados.

- Desempenho de Leitura (I/O): Usaremos a biblioteca time para cronometrar os milissegundos que o Python leva para carregar cada um desses arquivos de volta para a memória.


In [3]:
import pandas as pd
import numpy as np
import time
import os

#Vamos gerar um dataset fictício com 1 MILHÃO de linhas... Isso pode levar alguns segundos hihihihihi

# 1. Criando um conjunto de dados grande para o teste
numero_linhas = 1_000_000
dados = {#observe que todas as colunas serão geradas de maneira aleatória por meio na biblioteca numpy -> np.random
    'id_venda': range(numero_linhas),
    'categoria': np.random.choice(['Eletrônicos', 'Roupas', 'Alimentos', 'Livros', 'Móveis'], numero_linhas),
    'preco': np.random.uniform(10.0, 1500.0, numero_linhas),
    'quantidade': np.random.randint(1, 10, numero_linhas),
    'status': np.random.choice(['Concluído', 'Pendente', 'Cancelado'], numero_linhas)
}

df_vendas = pd.DataFrame(dados)


# 2. Salvando em CSV
inicio_csv = time.time()#aqui vamos usar biblioteca time para conseguir marcar o inicio e o fim do processo para então compara-los.
df_vendas.to_csv("vendas_armazenamento.csv", index=False)
tempo_escrita_csv = time.time() - inicio_csv #calculo do intervalo de tempo para salvar o csv

# 3. Salvando em Parquet
inicio_parquet = time.time()
df_vendas.to_parquet("vendas_armazenamento.parquet", engine='pyarrow')
tempo_escrita_parquet = time.time() - inicio_parquet#calculo do intervalo de tempo para salvar o parquet

# 4. Comparando o Tamanho
# os.path.getsize retorna o tamanho em bytes. Dividimos por (1024*1024) para transformar em Megabytes3
tamanho_csv_mb = os.path.getsize("vendas_armazenamento.csv") / (1024 * 1024)
tamanho_parquet_mb = os.path.getsize("vendas_armazenamento.parquet") / (1024 * 1024)

# 5. Comparando o Tempo de Leitura
inicio_leitura = time.time()
pd.read_csv("vendas_armazenamento.csv")
tempo_leitura_csv = time.time() - inicio_leitura

inicio_leitura = time.time()
pd.read_parquet("vendas_armazenamento.parquet", engine='pyarrow')
tempo_leitura_parquet = time.time() - inicio_leitura

# 6. Exibindo o Grande Resultado

print("RESULTADO DO COMPARATIVO: CSV vs PARQUET")
print("-" * 50)
print(f"TAMANHO NO DISCO:")
print(f"   CSV:     {tamanho_csv_mb:.2f} MB")
print(f"   Parquet: {tamanho_parquet_mb:.2f} MB (Aproximadamente {tamanho_csv_mb/tamanho_parquet_mb:.1f}x menor!)")
print("")
print(f"TEMPO DE LEITURA (Carregar na memória):")
print(f"   CSV:     {tempo_leitura_csv:.3f} segundos")
print(f"   Parquet: {tempo_leitura_parquet:.3f} segundos (Aproximadamente {tempo_leitura_csv/tempo_leitura_parquet:.1f}x mais rápido!)")
print("-" * 50)

RESULTADO DO COMPARATIVO: CSV vs PARQUET
--------------------------------------------------
TAMANHO NO DISCO:
   CSV:     45.06 MB
   Parquet: 13.06 MB (Aproximadamente 3.4x menor!)

TEMPO DE LEITURA (Carregar na memória):
   CSV:     0.594 segundos
   Parquet: 0.215 segundos (Aproximadamente 2.8x mais rápido!)
--------------------------------------------------


## 2. O Data Lake na Prática: Arquitetura Medalhão e DuckDB

Como vimos, a **Arquitetura Medalhão** organiza o Data Lake em três camadas (Bronze, Silver e Gold) de acordo com o nível de pureza e tratamento do dado. 

Para demonstrar isso, vamos simular um sistema de arquivos local criando pastas para cada camada. Em seguida, processaremos um dado de vendas (gerado na nossa origem OLTP) passando por todas as etapas, e usaremos o **DuckDB** (nosso banco OLAP in-process) para analisar os resultados finais usando SQL puro.

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

#1. Construindo a estrutura do Data Lake
# Criando as pastas simulando as camadas do Data Lake
pastas = ["datalake/bronze", "datalake/silver", "datalake/gold"]
for pasta in pastas:
    Path(pasta).mkdir(parents=True, exist_ok=True)
print("Pastas criadas com sucesso!\n")

#2. Simulando a extração de um sistema OLTP -> camda bronze
# Gerando um dado "sujo" simulando a extração de um sistema transacional
dados_brutos = {
    'id_pedido': [1, 2, 3, 4, 5],
    'data': ['2026-03-01', '2026-03-01', '2026-03-02', None, '2026-03-03'],
    'produto': ['  notebook ', 'MOUSE', 'teclado', 'monitor', 'MOUSE '],
    'valor': [3500.0, 150.0, 200.0, 800.0, 150.0]
}

df_bruto = pd.DataFrame(dados_brutos)

# Salvamos na Bronze exatamente como veio -> em csv
caminho_bronze = "datalake/bronze/pedidos_raw.csv"
df_bruto.to_csv(caminho_bronze, index=False)
print(f"Dados brutos salvos em: {caminho_bronze}")
display(df_bruto)

Pastas criadas com sucesso!

Dados brutos salvos em: datalake/bronze/pedidos_raw.csv


,id_pedido,data,produto,valor
0,1,2026-03-01,notebook,3500.0
1,2,2026-03-01,MOUSE,150.0
2,3,2026-03-02,teclado,200.0
3,4,None,monitor,800.0
4,5,2026-03-03,MOUSE,150.0


###  Camada Silver e o tratamento de Dados
Na camada **Silver**, o objetivo é limpar e padronizar. Vamos ler o arquivo da Bronze, remover linhas com datas vazias e padronizar o nome dos produtos. Em seguida, salvaremos no formato **Parquet**, que é colunar e muito mais eficiente para análises futuras.

In [14]:
# Lendo os dados brutos da Bronze
df_silver = pd.read_csv("datalake/bronze/pedidos_raw.csv")

# 1. Removendo valores ausentes
df_silver = df_silver.dropna(subset=['data'])

# 2. Padronizando textos 
df_silver['produto'] = df_silver['produto'].str.strip().str.lower()#removendo espaços em branco e deixando tudo minúsculo

# Salvando na camada Silver em formato Parquet
caminho_silver = "datalake/silver/pedidos_cleaned.parquet"
df_silver.to_parquet(caminho_silver, engine='pyarrow')
print(f"Dados limpos e salvos na Silver em: {caminho_silver}")
display(df_silver)

Dados limpos e salvos na Silver em: datalake/silver/pedidos_cleaned.parquet


,id_pedido,data,produto,valor
0,1,2026-03-01,notebook,3500.0
1,2,2026-03-01,mouse,150.0
2,3,2026-03-02,teclado,200.0
4,5,2026-03-03,mouse,150.0


### Camada Gold
A camada **Gold** é focada no usuário final, como analistas e diretorias. Aqui, não queremos ver pedidos individuais, mas sim indicadores prontos. Vamos criar uma tabela com o total de vendas por produto.

In [18]:
# Lendo da Silver
df_analise = pd.read_parquet("datalake/silver/pedidos_cleaned.parquet")

# 1. Agregando valor aos dados: faturamento total por produto
df_gold = df_analise.groupby('produto', as_index=False)['valor'].sum()
df_gold = df_gold.rename(columns={'valor': 'faturamento_total'})

# 2. Salvando na Gold
caminho_gold = "datalake/gold/faturamento_produto.parquet"
df_gold.to_parquet(caminho_gold, engine='pyarrow')
print(f"Indicadores de negócio salvos na Gold em: {caminho_gold}")
display(df_gold)

Indicadores de negócio salvos na Gold em: datalake/gold/faturamento_produto.parquet


,produto,faturamento_total
0,mouse,300.0
1,notebook,3500.0
2,teclado,200.0


## Consultas Analíticas (OLAP) com DuckDB
Agora que nosso Data Lake está organizado e os dados estão em formato Parquet, entra em cena o **DuckDB**. 
Como ele é um banco in-process, não precisamos subir um servidor pesado. Podemos usar a linguagem **SQL** diretamente no nosso notebook para consultar os arquivos Parquet que acabamos de criar nas camadas Silver e Gold, aproveitando a máxima velocidade do armazenamento colunar.

- OBS: para instalar o DuckDB, execute no terminal ou em uma célula: `!pip install duckdb`

In [19]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [22]:
import duckdb

#consulta SQL OLAP diretamente no arquivo Parquet da camada Silver

# 1. O DuckDB permite fazer o SELECT apontando diretamente para o caminho do arquivo.
consulta_silver = """
    SELECT data, COUNT(id_pedido) as qtd_pedidos, SUM(valor) as total_dia
    FROM 'datalake/silver/pedidos_cleaned.parquet'
    GROUP BY data
    ORDER BY data;
"""

# 2. Executando e convertendo o resultado para um DataFrame Pandas para visualização
resultado_olap = duckdb.query(consulta_silver).df()
display(resultado_olap)

# 3. Executando consulta na camada Gold -> buscando produtos com faturamento > 200
consulta_gold = """
    SELECT produto, faturamento_total
    FROM 'datalake/gold/faturamento_produto.parquet'
    WHERE faturamento_total > 200
    ORDER BY faturamento_total DESC;
"""

resultado_gold = duckdb.query(consulta_gold).df()#transformando em data freame para visualizar
display(resultado_gold)

,data,qtd_pedidos,total_dia
0,2026-03-01,2,3650.0
1,2026-03-02,1,200.0
2,2026-03-03,1,150.0


,produto,faturamento_total
0,notebook,3500.0
1,mouse,300.0


## 3. Padrão ELT e Staging com DuckDB

Diferente do ETL tradicional, onde tratamos os dados antes de salvar, no padrão **ELT** nós primeiro carregamos (Load) os dados brutos para dentro do nosso ambiente e, em seguida, usamos o poder de processamento do próprio banco de dados para transformá-los (Transform).

Para organizar esse processo, o DuckDB permite criar schemas. Neste exemplo, vamos criar três schemas:
- 1. **Raw:** recebe os dados exatamente como vieram da fonte.
- 2. **Staging:** recebe os dados com limpezas técnicas, como remoção de duplicados, correção de tipos...
- 3. **Analytics:** recebe os dados prontos para o uso, com regras aplicadas.

In [ ]:
import duckdb
import pandas as pd

# Conectando ao DuckDB (usaremos um banco em memória para o exemplo)
con = duckdb.connect(database=':memory:')

# Aqui estamos criando a os schemas
con.execute("CREATE SCHEMA IF NOT EXISTS raw;")
con.execute("CREATE SCHEMA IF NOT EXISTS staging;")
con.execute("CREATE SCHEMA IF NOT EXISTS analytics;")


# Criamos um dataset sujo: ID duplicado, valores nulos e tipo de dado errado.
dados_extraidos = pd.DataFrame({
    'id_cliente': [1, 2, 2, 3, 4], 
    'nome': ['Ana', 'Bruno', 'Bruno', 'Carlos', None], 
    'idade': ['28', '35', '35', 'N/A', '40'] 
})

# O DuckDB consegue ler o DataFrame do Pandas e criar a tabela direto no schema 'raw'
con.execute("CREATE TABLE raw.clientes AS SELECT * FROM dados_extraidos")

display(con.query("SELECT * FROM raw.clientes").df())#imprimindo para visualização com o auxulio do pandas

,id_cliente,nome,idade
0,1,Ana,28
1,2,Bruno,35
2,2,Bruno,35
3,3,Carlos,N/A
4,4,None,40


### A etapa de transformação

Com os dados já carregados no banco, usamos a linguagem SQL para resolver os problemas de qualidade. 

- 1. Moveremos os dados da `raw` para a `staging`, removendo duplicados, filtrando valores nulos e corrigindo a coluna de idade (que veio como texto).
- 2. Depois, moveremos da `staging` para a `analytics`, criando uma classificação de negócio pronta para o consumo de ferramentas de BI.

In [ ]:

# 1. Limpeza removemos nulos, tratamos o 'N/A', convertemos para INTEIRO e removemos duplicados
query_staging = """
    CREATE TABLE staging.clientes_tratados AS
    SELECT DISTINCT
        CAST(id_cliente AS INTEGER) AS id_cliente,
        UPPER(nome) AS nome,
        CAST(CASE WHEN idade = 'N/A' THEN NULL ELSE idade END AS INTEGER) AS idade
    FROM raw.clientes
    WHERE nome IS NOT NULL;
"""
con.execute(query_staging)#executando a query
display(con.query("SELECT * FROM staging.clientes_tratados").df())#imprimindo com pandas depois do tratamento

# 2. Por último, para mover para o ultimo schema -> analytics, vamos aplicar a regra de negócio, por exemplo: categoria baseada na idade
query_analytics = """
    CREATE TABLE analytics.clientes_perfil AS
    SELECT 
        id_cliente,
        nome,
        idade,
        CASE 
            WHEN idade < 30 THEN 'Jovem'
            WHEN idade >= 30 THEN 'Adulto'
            ELSE 'Não Informado'
        END AS faixa_etaria
    FROM staging.clientes_tratados;
"""
con.execute(query_analytics)
display(con.query("SELECT * FROM analytics.clientes_perfil").df())

,id_cliente,nome,idade
0,1,ANA,28
1,2,BRUNO,35
2,3,CARLOS,<NA>


,id_cliente,nome,idade,faixa_etaria
0,1,ANA,28,Jovem
1,2,BRUNO,35,Adulto
2,3,CARLOS,<NA>,Não Informado
